In [98]:
import pandas as pd
import numpy as np

# S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"] #  on récupère l'url du stockage s3

# # on peut désormais instancier le filesystem en précisant l'url du stockage S3

# fs = s3fs.S3FileSystem(endpoint_url= S3_ENDPOINT_URL)


import psutil


# Get total memory in bytes
total_memory = psutil.virtual_memory().total

# Convert bytes to GB (optional, for readability)
total_memory_in_gb = total_memory / (1024 ** 3)

print(f"Total RAM: {total_memory_in_gb:.2f} GB")
import os

from dotenv import load_dotenv


Total RAM: 31.43 GB


In [99]:
df = pd.read_parquet("data/JOCAS/small/jocas_small_all_half_sirenised_embed_job_title.parquet")

In [100]:
import numpy as np

def cosine_similarity_matrix(X: np.ndarray) -> np.ndarray:
    """
    Compute cosine similarity between all rows of X.
    X: (n, d)
    Returns: (n, n)
    """
    X_norm = X / np.linalg.norm(X, axis=1, keepdims=True)
    return X_norm @ X_norm.T

In [101]:
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")


# 1. Annotation Training set

In [102]:
from openai import OpenAI

client = OpenAI(api_key=api_key)

In [103]:
client = OpenAI()

In [104]:
import pandas as pd
# df = pd.read_parquet("data/annotation/training_set_labeled.parquet")


Drop job postings of stage

In [105]:
import pandas as pd
import re
import unicodedata

def clean_job_title(text):
    if pd.isna(text):
        return pd.NA

    text = str(text).strip().lower()

    # Enlever les accents : développeur -> developpeur, île -> ile
    text = ''.join(
        c for c in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(c)
    )

    text = text.replace("/", " ")
    text = text.replace("-", " ")

    text = re.sub(r"[^\w\s]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

df["job_title_clean"] = df["job_title"].apply(clean_job_title)

mask_stage = df["job_title_clean"].str.contains(r"\bstage\b", na=False)
df_without_stage = df[~mask_stage]

In [107]:
df_without_stage.to_parquet("data/JOCAS/jocas_small_all_half_sirenised_embed_job_title_drop_stage.parquet")

In [62]:
df['description_clean'].loc[5]

'rejoignez aread ! pour accompagner sa croissance sur la région grand est, aread recherche un(e) consultant(e) expérimenté(e) en financement de l\'innovation en cdi avec une forte culture technologique en sciences physiques et électronique. objectif de la mission : réaliser les dossiers de financement de projets r&d et d\'innovation du portefeuille clients du grand est, et développer l\'activité sur ce territoire. détail de la mission : vous intégrerez l\'équipe d\'un cabinet de conseil spécialisé en financement de la r&d et de l\'innovation. créé en 2003, il est implanté à strasbourg (siège), paris, lyon, nantes, bordeaux, tours, caen et toulouse. il accompagne ses clients dans la recherche de financements et aides publics afin de leur permettre de développer de nouveaux projets dans l\'innovation, la r&d et leur développement en général. la société est opqcm et référencée "cabinet en cir/cii". la principale mission sera d\'assurer pour le compte des clients régionaux des missions de 

# 2. LLM Fine tuning

# 3. High scale inference